# Question 3: Abstractive summarisation with a Transformer written from scratch

DSL 504 NLP Group Assignment 2. We train a Transformer encoder-decoder ("Attention Is All You Need") on
CNN / Daily Mail 3.0.0 to turn a news article into a short multi-sentence summary, then evaluate it with
ROUGE on the full test set, look at examples and plot encoder-decoder attention.

The Transformer has an optional **copy (pointer) switch** (on by default): at every step the decoder can either
generate a word from the vocabulary or copy a word from the article through its encoder-decoder attention. This
is what keeps names, places and numbers tied to the article instead of invented.

The from-scratch model is in `q3_model.py`; data handling in `q3_data.py`; loss, schedule and batched
generation in `q3_train.py`; ROUGE and checks in `q3_eval.py`; `q3_report.py` writes `REPORT.md` from the
results. This notebook runs everything in order:

| Section | What happens |
|---|---|
| 1 | Setup and configuration |
| 2 | Data: load, lowercase, tokenise, truncate, statistics |
| 3 | Model and parameter count |
| 4 | Checks before training (masks, padding, loss, banned layers, memorising real pairs) |
| 5 | Training with teacher forcing, Noam schedule and early stopping on validation loss |
| 6 | Restore the best checkpoint; choose the decoding setting on validation |
| 7 | ROUGE-1/2/L/Lsum on all 11,490 test articles, with Lead-3 for comparison |
| 8 | Qualitative examples, including failure cases |
| 9 | Encoder-decoder attention heatmaps |
| 10 | Report, deliverables checklist and output ZIP |

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **on**, then *Save Version → Save & Run All*.
The default configuration trains for 3.25 h; the whole notebook takes about 4 to 4.5 h.
Set `SMOKE = True` for a two-minute check on CPU (numbers from a smoke run are not results).

## 1. Setup

In [ ]:
# Find the repository (shared `common/` helpers + the Question-3 code).
# Locally we are already inside it. On Kaggle it is either an uploaded dataset or a git clone.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Akshats-git/NLP-Group-Assignment-2.git"
REPO_BRANCH = "main"   # change if the Question-3 code is on another branch


def find_repo():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "common").is_dir() and (folder / "Question-3" / "q3_model.py").is_file():
            return folder
    working = Path("/kaggle/working")
    target = (working if working.exists() else Path.cwd()) / "NLP-Group-Assignment-2"
    if target.exists():
        return target
    for model_file in Path("/kaggle/input").glob("**/Question-3/q3_model.py"):   # uploaded Kaggle dataset
        shutil.copytree(model_file.parents[1], target)   # copy, because /kaggle/input is read-only
        return target
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(target)], check=True)
    return target


REPO = find_repo()
os.chdir(REPO)
sys.path[:0] = [str(REPO), str(REPO / "Question-3")]
print("repository:", REPO)

In [ ]:
%pip install -q rouge-score==0.1.2

In [ ]:
import json, math, time, zipfile

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from common import (EarlyStopping, History, autocast, count_parameters, dataloader_kwargs, format_count,
                    get_device, grad_scaler, save_json, save_jsonl, save_run_info, set_seed,
                    teacher_forcing_pair)
from common.plotting import attention_window, plot_attention, plot_grouped_bars, plot_history, savefig

import q3_data, q3_eval, q3_model, q3_report, q3_train

SMOKE = False   # True: tiny model + small subsets, only to check the notebook runs (CPU is fine)

CFG = dict(
    run_name="q3_transformer_seed42",
    seed=42,
    # data
    src_len=400,              # article tokens kept, incl. [CLS]/[SEP] (brief: 400-512)
    tgt_len=128,              # summary tokens kept, incl. [CLS]/[SEP] (brief: about 128)
    train_size=None,          # None = all 287,113 training pairs
    val_size=None,            # None = all 13,368 validation pairs (used for the validation loss)
    test_size=None,           # None = all 11,490 test pairs. Never reduce this for real results.
    # model (brief: d_model 256-512, 3-6 layers per stack, 4-8 heads)
    d_model=512, n_heads=8, n_enc_layers=4, n_dec_layers=4, d_ff=2048, dropout=0.1,
    copy_mechanism=True,      # pointer / copy switch over the article (See et al., 2017); False = plain Transformer
    # training
    batch_size=32,            # pairs per GPU per step (2 x T4 -> 64 pairs per step)
    multi_gpu=True,           # use both Kaggle T4s with nn.DataParallel
    warmup_steps=4000, label_smoothing=0.1, clip_norm=1.0,
    eval_every=3000,          # steps between validation checks
    patience=3,               # early stopping: stop after 3 checks without a lower validation loss
    train_hours=3.25,         # time budget for training (summed over sessions when a run is resumed)
    max_epochs=10,
    resume=True,              # continue from outputs/checkpoints/last.pt if an earlier run was interrupted
    # generation and evaluation
    max_summary_len=128,      # greedy decoding stops at [SEP] or after this many tokens
    eval_batch_size=64,
    val_monitor_size=256,     # validation articles decoded at every check, to follow ROUGE during training
    decode_choice_size=1000,  # validation articles used to choose plain greedy vs greedy + trigram blocking
    attention_articles=256,   # test articles used for the attention-by-position statistic
)
if SMOKE:
    CFG.update(run_name="smoke", train_size=1500, val_size=200, test_size=96, d_model=64, n_heads=4,
               n_enc_layers=2, n_dec_layers=2, d_ff=128, batch_size=16, warmup_steps=100, eval_every=60,
               train_hours=0.06, max_epochs=3, max_summary_len=60, eval_batch_size=32, val_monitor_size=32,
               decode_choice_size=48, attention_articles=32)

ON_KAGGLE = Path("/kaggle/working").exists()
Q3 = REPO / "Question-3"
OUT = Q3 / ("outputs_smoke" if SMOKE else "outputs")          # results, figures, checkpoints
RESULTS, FIGURES, CKPT = OUT / "results", OUT / "figures", OUT / "checkpoints"
DATA = Path("/tmp/q3_data") if ON_KAGGLE else Q3 / "data"     # token cache; never submitted
for folder in (RESULTS, FIGURES, CKPT, DATA):
    folder.mkdir(parents=True, exist_ok=True)

set_seed(CFG["seed"])
device = get_device()
n_gpus = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0) if n_gpus else "CPU"
use_gpus = n_gpus if (CFG["multi_gpu"] and n_gpus > 1) else 1
print(f"device: {device}, GPUs: {n_gpus} ({gpu_name}), smoke run: {SMOKE}")

## 2. Data

CNN / Daily Mail 3.0.0 from Hugging Face (`abisee/cnn_dailymail` is where the `cnn_dailymail` dataset lives
now; same data and splits). Preprocessing, as the brief asks:

* **lowercase** articles and highlights (the tokenizer is uncased);
* **sub-word tokenizer**: pre-trained `bert-base-uncased` WordPiece, one shared vocabulary for articles and
  summaries, plus one new token `[NL]` that stands for the newline between highlight sentences (so the model
  learns sentence breaks and we can write its output one sentence per line for ROUGE-Lsum);
* `[CLS]` = start token, `[SEP]` = end token, `[PAD]` = padding (id 0);
* truncate articles to **400** and summaries to **128** tokens (a cut sequence still ends with `[SEP]`);
* decoder input = summary shifted right, target = summary shifted left; batches are padded to their longest row.

In [ ]:
dataset = q3_data.load_cnn_dailymail()
tokenizer = q3_data.load_tokenizer()
SPECIAL = q3_data.special_ids(tokenizer)
VOCAB_SIZE = len(tokenizer)

sizes = {name: len(dataset[name]) for name in ("train", "validation", "test")}
assert sizes == {"train": 287113, "validation": 13368, "test": 11490}, sizes
print("pairs per split:", sizes)
print("vocabulary size:", VOCAB_SIZE, "| special ids:", SPECIAL)

In [ ]:
# One training example through the whole preprocessing pipeline
example = dataset["train"][0]
summary_text = q3_data.prepare_summary(example["highlights"])
summary_ids = q3_data.truncate(tokenizer(summary_text)["input_ids"], CFG["tgt_len"])
decoder_input, target = teacher_forcing_pair(torch.tensor([summary_ids]))

print("highlights (raw):\n" + example["highlights"])
print("\nafter lowercasing + [NL]:\n" + summary_text)
print("\nWordPiece tokens:", tokenizer.convert_ids_to_tokens(summary_ids)[:24], "...")
print("decoder input   :", tokenizer.convert_ids_to_tokens(decoder_input[0])[:8], "...")
print("target          :", tokenizer.convert_ids_to_tokens(target[0])[:8], "...")
print("\nids back to text (one sentence per line):\n" + q3_data.ids_to_text(summary_ids[1:], tokenizer, SPECIAL))

In [ ]:
def subset(split, size, seed):
    # A fixed random subset (same every run), or the whole split.
    if size is None or size >= len(split):
        return split
    keep = np.random.default_rng(seed).permutation(len(split))[:size]
    return split.select(np.sort(keep))


splits = {
    "train": subset(dataset["train"], CFG["train_size"], CFG["seed"]),
    "validation": subset(dataset["validation"], CFG["val_size"], CFG["seed"]),
    "test": subset(dataset["test"], CFG["test_size"], CFG["seed"]),
}


def tokenize_split(name):
    # Token ids for one split, cached as .npz so a rerun skips this step.
    split = splits[name]
    cache = DATA / f"{name}_{len(split)}_{CFG['src_len']}_{CFG['tgt_len']}.npz"
    if cache.exists():
        with np.load(cache) as arrays:
            return {key: arrays[key] for key in arrays.files}
    src, src_full = q3_data.encode_texts(split, "article", q3_data.prepare_article, tokenizer,
                                         CFG["src_len"], desc=f"{name} articles")
    tgt, tgt_full = q3_data.encode_texts(split, "highlights", q3_data.prepare_summary, tokenizer,
                                         CFG["tgt_len"], desc=f"{name} summaries")
    np.savez(cache, src=src, tgt=tgt, src_full=src_full, tgt_full=tgt_full)
    return {"src": src, "tgt": tgt, "src_full": src_full, "tgt_full": tgt_full}


t0 = time.time()
data = {name: tokenize_split(name) for name in splits}
print(f"tokenised in {time.time() - t0:.0f} s")

# Raw text, kept for ROUGE (references keep their newlines) and for the examples
val_refs = list(splits["validation"]["highlights"])
test_refs = list(splits["test"]["highlights"])
test_articles = list(splits["test"]["article"])
test_ids = list(splits["test"]["id"])

In [ ]:
data_info = {
    "dataset": f"{q3_data.DATASET_NAME} {q3_data.DATASET_VERSION}",
    "tokenizer": q3_data.TOKENIZER_NAME,
    "vocab_size": VOCAB_SIZE,
    "lowercase": True,
    "special_tokens": {"start": "[CLS]", "end": "[SEP]", "pad": "[PAD]", "sentence_break": q3_data.NEWLINE_TOKEN},
    "src_len": CFG["src_len"], "tgt_len": CFG["tgt_len"],
    "pairs": {name: len(split) for name, split in splits.items()},
    "lengths": {name: {"article": q3_data.length_stats(arrays["src_full"], CFG["src_len"]),
                       "summary": q3_data.length_stats(arrays["tgt_full"], CFG["tgt_len"])}
                for name, arrays in data.items()},
}
save_json(data_info, RESULTS / "data_info.json")

print(f"{'split':<11}{'text':<9}{'median':>8}{'p90':>8}{'max':>7}{'limit':>7}{'truncated':>11}")
for name in data:
    for kind in ("article", "summary"):
        s = data_info["lengths"][name][kind]
        print(f"{name:<11}{kind:<9}{s['median']:>8.0f}{s['p90']:>8.0f}{s['max']:>7}{s['limit']:>7}"
              f"{s['pct_truncated']:>10.1f}%")

In [ ]:
train_set = q3_data.SummaryDataset(data["train"]["src"], data["train"]["tgt"])
val_set = q3_data.SummaryDataset(data["validation"]["src"], data["validation"]["tgt"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"   # tokenising is done; avoids a warning when workers fork
batch_size_total = CFG["batch_size"] * use_gpus
loader_args = dict(collate_fn=q3_data.collate, num_workers=2, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_set, batch_size=batch_size_total, shuffle=True, drop_last=True,
                          **loader_args, **dataloader_kwargs(CFG["seed"]))
val_loader = DataLoader(val_set, batch_size=CFG["eval_batch_size"] * use_gpus, shuffle=False, **loader_args)

# Fixed random validation samples for monitoring during training and for the decoding choice
val_order = np.random.default_rng(CFG["seed"]).permutation(len(val_set))
print(f"{len(train_loader):,} training steps per epoch with {batch_size_total} pairs per step")

## 3. Model

`q3_model.TransformerSummarizer` (read `q3_model.py` for the code). Post-norm layers exactly as in the brief,
`LayerNorm(x + Dropout(Sublayer(x)))`; sinusoidal positional encodings; one embedding matrix shared by encoder
input, decoder input and the output layer, scaled by sqrt(d_model) as in the paper.

**Copy switch.** A plain Transformer has to produce every name and number from its 30k-word softmax, and early in
training it writes fluent but unrelated "summary-sounding" text. With `copy_mechanism=True` a small gate computes
`p_gen = sigmoid(W [decoder state; article context])` and the next-word probability becomes

`P(w) = p_gen · P_vocab(w) + (1 − p_gen) · (attention on the article positions holding w)`,

using the last decoder layer's cross-attention averaged over heads. It adds only 1,025 parameters.

In [ ]:
model_args = dict(pad_id=SPECIAL.pad, d_model=CFG["d_model"], n_heads=CFG["n_heads"],
                  n_enc_layers=CFG["n_enc_layers"], n_dec_layers=CFG["n_dec_layers"], d_ff=CFG["d_ff"],
                  dropout=CFG["dropout"], max_len=max(CFG["src_len"], CFG["max_summary_len"] + 1, 512),
                  copy_mechanism=CFG["copy_mechanism"])
model = q3_model.TransformerSummarizer(VOCAB_SIZE, **model_args).to(device)
params = count_parameters(model)

print(model.encoder_layers[0])
print(model.decoder_layers[0])
print(f"\n{CFG['n_enc_layers']} encoder + {CFG['n_dec_layers']} decoder layers, d_model {CFG['d_model']}, "
      f"{CFG['n_heads']} heads, d_ff {CFG['d_ff']}, copy mechanism {'on' if CFG['copy_mechanism'] else 'off'}")
print(f"parameters: {params['total']:,} ({format_count(params['total'])}), of which "
      f"{model.embed.weight.numel():,} in the shared embedding / output matrix")
save_run_info(RESULTS, CFG, seed=CFG["seed"], parameters=params, gpus_used=use_gpus, gpu=gpu_name)

## 4. Checks before training

Cheap tests for the pitfalls listed in the brief, run on the real code before spending GPU hours:

1. no forbidden building block (`nn.Transformer*`, `nn.MultiheadAttention`, `scaled_dot_product_attention`);
2. **causal mask**: changing later summary tokens must not change the outputs at earlier positions;
3. **padding masks**: extra padding on the article or the summary must not change the outputs;
4. **loss**: padding positions get no loss and no gradient;
5. **end to end**: a small model trained on 8 real training pairs must reproduce them exactly with greedy
   decoding (data, teacher forcing, masks, loss and generation all fit together).

In [ ]:
import ast

checks = {}

# 1. No forbidden building blocks, neither in the source code nor in the built model
banned = {"Transformer", "TransformerEncoder", "TransformerDecoder", "TransformerEncoderLayer",
          "TransformerDecoderLayer", "MultiheadAttention", "scaled_dot_product_attention"}
tree = ast.parse((Q3 / "q3_model.py").read_text())
names_used = {n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)} | \
             {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)}
banned_modules = (nn.Transformer, nn.TransformerEncoder, nn.TransformerDecoder, nn.TransformerEncoderLayer,
                  nn.TransformerDecoderLayer, nn.MultiheadAttention)
checks["no_banned_layers"] = not (names_used & banned) and \
    not any(isinstance(m, banned_modules) for m in model.modules())

# A small copy of the architecture in fp32 on the CPU for the mask tests. The model returns
# (vocabulary logits, p_gen, copy attention); the last two are None without the copy mechanism.
torch.manual_seed(0)
small = q3_model.TransformerSummarizer(VOCAB_SIZE, d_model=64, n_heads=4, n_enc_layers=2, n_dec_layers=2,
                                       d_ff=128, dropout=0.0, copy_mechanism=CFG["copy_mechanism"]).eval()


def close(a, b):
    return all(x is None or torch.allclose(x, y, atol=1e-5) for x, y in zip(a, b))


def first_positions(outputs, n):
    return tuple(None if x is None else x[:, :n] for x in outputs)


src = torch.randint(5, VOCAB_SIZE, (2, 30))
tgt = torch.randint(5, VOCAB_SIZE, (2, 12))
with torch.no_grad():
    base = small(src, tgt)

    # 2. Causal mask: change tokens 6.. of the summary; positions 0..5 must not change
    changed = tgt.clone()
    changed[:, 6:] = 7
    after = small(src, changed)
    checks["causal_mask"] = close(first_positions(base, 6), first_positions(after, 6)) and \
        not torch.allclose(base[0][:, 6:], after[0][:, 6:], atol=1e-5)

    # 3a. Source padding: 10 extra [PAD] tokens after the article change nothing and get zero attention
    padded_src = torch.cat([src, torch.full((2, 10), SPECIAL.pad)], dim=1)
    padded = small(padded_src, tgt)
    memory, src_mask = small.encode(padded_src)
    _, cross = small.decode(tgt, memory, src_mask)
    copy_part_ok = padded[2] is None or (torch.allclose(base[2], padded[2][..., :30], atol=1e-5)
                                         and padded[2][..., 30:].max().item() < 1e-6)
    checks["source_padding_mask"] = close(base[:2], padded[:2]) and copy_part_ok and \
        cross[..., -10:].max().item() < 1e-6

    # 3b. Target padding: padding after the summary does not change the real positions
    padded_tgt = torch.cat([tgt, torch.full((2, 5), SPECIAL.pad)], dim=1)
    checks["target_padding_mask"] = close(base, first_positions(small(src, padded_tgt), 12))

# 4. Loss ignores padding: same loss as cross_entropy(ignore_index=pad) and no gradient at padding
logits = torch.randn(2, 6, VOCAB_SIZE, requires_grad=True)
targets = torch.tensor([[5, 6, 7, 0, 0, 0], [8, 9, 10, 11, 12, 0]])
loss, _, n_tokens = q3_train.seq2seq_loss(logits, targets, SPECIAL.pad)
reference = nn.functional.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1), ignore_index=0)
loss.backward()
checks["loss_ignores_padding"] = n_tokens.item() == 8 and torch.allclose(loss, reference) and \
    logits.grad[targets == 0].abs().max().item() == 0

# 5. Memorise 8 real training pairs, then reproduce them with greedy decoding
torch.manual_seed(0)
toy = q3_model.TransformerSummarizer(VOCAB_SIZE, d_model=128, n_heads=4, n_enc_layers=2, n_dec_layers=2,
                                     d_ff=256, dropout=0.0, copy_mechanism=CFG["copy_mechanism"]).to(device)
toy_opt = torch.optim.Adam(toy.parameters(), lr=1e-3)
toy_src, toy_tgt = (t.to(device) for t in q3_data.collate([train_set[i] for i in range(8)]))
toy_in, toy_out = teacher_forcing_pair(toy_tgt)
for _ in range(300):
    toy.train()
    toy_loss, _, _ = q3_train.seq2seq_loss(toy(toy_src, toy_in), toy_out, SPECIAL.pad, src=toy_src)
    toy_opt.zero_grad()
    toy_loss.backward()
    toy_opt.step()
generated, _ = q3_model.greedy_summarize(toy, toy_src, SPECIAL.bos, SPECIAL.eos, SPECIAL.pad,
                                         max_len=toy_tgt.size(1), banned_ids=[SPECIAL.pad, SPECIAL.bos])
exact = 0
for gen_row, ref_row in zip(generated.tolist(), toy_out.tolist()):
    ref_row = ref_row[: ref_row.index(SPECIAL.eos) + 1]
    exact += gen_row[: len(ref_row)] == ref_row
checks["memorise_8_real_pairs"] = exact == 8
del toy, toy_opt

save_json(checks, RESULTS / "sanity_checks.json")
for name, ok in checks.items():
    print(f"{'pass' if ok else 'FAIL'}  {name}")
print(f"(memorisation: {exact}/8 summaries reproduced exactly, final loss {toy_loss.item():.4f})")
assert all(checks.values()), "a check failed: fix it before training"

## 5. Training

* **Teacher forcing**: the decoder gets the gold summary shifted right and predicts the summary shifted left, all
  positions in one parallel pass; the causal mask hides later tokens.
* **Loss**: cross-entropy over real (non-padding) target tokens, with label smoothing 0.1 for training. The plain
  cross-entropy (no smoothing) is logged for train and validation and is what early stopping uses.
* **Optimiser**: Adam with β = (0.9, 0.98), ε = 1e-9 and the paper's schedule
  `lr = d_model^-0.5 · min(step^-0.5, step · warmup^-1.5)`, gradient clipping at 1.0, mixed precision on the GPU.
* **Early stopping**: every `eval_every` steps we compute the validation loss over all validation pairs, keep the
  best checkpoint on disk and stop after `patience` checks without improvement, or when the time budget runs out.
  At every check we also decode a fixed sample of validation articles (no teacher forcing) to follow ROUGE.
* **Is it reading the article?** At every check the same validation summaries are also scored against the *wrong*
  article (each batch rotated by one). If the model uses the article, the loss with the right article is clearly
  lower. Early in training (before and around the end of warm-up) the decoder mostly writes generic,
  summary-sounding text; the gap and ROUGE-2 show when it starts to follow the article.
* **Resumable**: after every check the whole training state (weights, optimiser, schedule, early stopping,
  history, steps and time used) is saved to `outputs/checkpoints/last.pt`. If the run is interrupted, running the
  notebook again continues from the last check instead of starting over; at most one check interval is lost.
  On Kaggle, attach the output of the interrupted version as input and it is picked up from there.
  If training had already finished, the notebook goes straight to evaluation.

In [ ]:
set_seed(CFG["seed"])   # same dropout randomness no matter which checks ran above
train_model = nn.DataParallel(model) if use_gpus > 1 else model   # splits each batch over the GPUs

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
# LambdaLR multiplies the base lr (1.0) by noam_lr, so the lr follows the paper's formula exactly
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: q3_train.noam_lr(step + 1, CFG["d_model"], CFG["warmup_steps"]))
scaler = grad_scaler(device)                     # loss scaling for fp16 on the T4; does nothing elsewhere
stopper = EarlyStopping(mode="min", patience=CFG["patience"], checkpoint_path=CKPT / "best.pt")
history = History()

# Resume after an interruption. last.pt holds the full training state from the last validation check.
RESUME_FILE = CKPT / "last.pt"
if CFG["resume"] and not RESUME_FILE.exists():
    # On Kaggle: the output of an earlier, interrupted version, attached with "Add Input"
    for found in sorted(Path("/kaggle/input").glob("**/Question-3/outputs/checkpoints/last.pt")):
        shutil.copy(found, RESUME_FILE)
        if (found.parent / "best.pt").exists():
            shutil.copy(found.parent / "best.pt", CKPT / "best.pt")
        print("copied the checkpoint of an earlier run from", found.parent)
        break
progress = {"step": 0, "examples_seen": 0, "train_seconds": 0.0, "stop_reason": None}
if CFG["resume"] and RESUME_FILE.exists():
    progress = q3_train.load_training_state(RESUME_FILE, model, optimizer, scheduler, scaler, stopper, history)
    print(f"RESUMING: {progress['step']:,} steps and {progress['train_seconds'] / 3600:.2f} h of training already done"
          + (f"; training had finished ({progress['stop_reason']}), going to evaluation" if progress["stop_reason"] else ""))

monitor_idx = val_order[: CFG["val_monitor_size"]]
monitor_src = data["validation"]["src"][monitor_idx]
monitor_tgt = data["validation"]["tgt"][monitor_idx]
monitor_refs = [val_refs[i] for i in monitor_idx]
print("reference of the validation article printed at each check:\n" + monitor_refs[0])


def validation_check(step, epochs_done, train_nll, elapsed):
    val_loss = q3_train.evaluate_loss(train_model, val_loader, SPECIAL.pad, device)
    preds = q3_train.summarize_arrays(model, monitor_src, tokenizer, SPECIAL, device,
                                      CFG["eval_batch_size"], CFG["max_summary_len"])
    rouge = q3_eval.rouge_means(q3_eval.rouge_per_example(preds, monitor_refs))
    article = q3_train.right_vs_wrong_article(model, monitor_src, monitor_tgt, SPECIAL.pad, device,
                                              CFG["eval_batch_size"])
    history.log(step=step, epoch=round(epochs_done, 3), train_loss=train_nll, val_loss=val_loss,
                val_ppl=math.exp(val_loss), val_rouge1=rouge["rouge1"], val_rouge2=rouge["rouge2"],
                val_rougeLsum=rouge["rougeLsum"], loss_right_article=article["right"],
                loss_wrong_article=article["wrong"], p_gen=article["p_gen"],
                lr=optimizer.param_groups[0]["lr"], time_s=round(elapsed))
    improved = stopper.step(val_loss, model)     # saves best.pt when the validation loss improves
    print(f"step {step:>6,} | epoch {epochs_done:4.2f} | train loss {train_nll:.3f} | val loss {val_loss:.3f} | "
          f"val R-1/2/Lsum {rouge['rouge1']:.1f}/{rouge['rouge2']:.1f}/{rouge['rougeLsum']:.1f} | "
          f"{elapsed / 60:.0f} min{'  <- best' if improved else ''}")
    print(f"    reads the article? loss with right article {article['right']:.2f} vs wrong article "
          f"{article['wrong']:.2f} (gap {article['wrong'] - article['right']:.2f})"
          + (f", p_gen {article['p_gen']:.2f}" if article["p_gen"] is not None else ""))
    print("    generated:", preds[0].replace("\n", " | ")[:300])

In [ ]:
step, examples_seen, stop_reason = progress["step"], progress["examples_seen"], progress["stop_reason"]
already_finished = stop_reason is not None
nll_sum = torch.zeros((), device=device)          # training NLL since the last check (token weighted)
token_count = torch.zeros((), device=device)
start = time.time() - progress["train_seconds"]   # time used in earlier sessions counts towards the budget
session_start, session_steps, session_examples = time.time(), 0, 0
if step > 0:
    train_loader.generator.manual_seed(CFG["seed"] + step)   # a fresh shuffle after resuming


def save_progress():
    # Full training state after a check, so an interruption loses at most one check interval
    q3_train.save_training_state(RESUME_FILE, model, optimizer, scheduler, scaler, stopper, history,
                                 {"step": step, "examples_seen": examples_seen,
                                  "train_seconds": time.time() - start, "stop_reason": stop_reason})
    history.save(RESULTS / "history.json")


for epoch in range(examples_seen // len(train_set) + 1, CFG["max_epochs"] + 1):
    if stop_reason:                                # training already finished in an earlier session
        break
    train_model.train()
    for src, tgt in train_loader:
        src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
        tgt_in, tgt_out = teacher_forcing_pair(tgt)   # [CLS] y1 .. yn  ->  y1 .. yn [SEP]
        with autocast(device):
            outputs = train_model(src, tgt_in)         # word distribution for every summary position at once
        loss, batch_nll, batch_tokens = q3_train.seq2seq_loss(outputs, tgt_out, SPECIAL.pad,
                                                              CFG["label_smoothing"], src=src)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_norm"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step += 1
        examples_seen += src.size(0)
        session_steps += 1
        session_examples += src.size(0)
        nll_sum += batch_nll
        token_count += batch_tokens
        elapsed = time.time() - start
        if session_steps == 200:
            rate = session_examples / (time.time() - session_start)
            print(f"speed: {rate:.0f} pairs/s -> about {rate * CFG['train_hours'] * 3600 / len(train_set):.1f} "
                  f"epochs in the {CFG['train_hours']} h budget")
        out_of_time = elapsed > CFG["train_hours"] * 3600
        if step % CFG["eval_every"] == 0 or out_of_time:
            validation_check(step, examples_seen / len(train_set), (nll_sum / token_count).item(), elapsed)
            nll_sum.zero_()
            token_count.zero_()
            train_model.train()
            out_of_time = time.time() - start > CFG["train_hours"] * 3600   # the check itself takes time
            if stopper.should_stop:
                stop_reason = f"early stopping (no better validation loss in {CFG['patience']} checks)"
            elif out_of_time:
                stop_reason = f"time budget of {CFG['train_hours']} h reached"
            save_progress()
            if stop_reason:
                break
    if stop_reason:
        break

if stop_reason is None:
    stop_reason = f"maximum of {CFG['max_epochs']} epochs reached"
    if token_count.item() > 0:            # the last steps have not been validated yet
        validation_check(step, examples_seen / len(train_set), (nll_sum / token_count).item(), time.time() - start)
    save_progress()

train_seconds = progress["train_seconds"] if already_finished else time.time() - start
history.save(RESULTS / "history.json")
print(f"\nstopped: {stop_reason} after {step:,} steps ({train_seconds / 3600:.2f} h)")

In [ ]:
best = history.best("val_loss")
training_summary = {
    "steps": step, "examples_seen": examples_seen, "epochs": examples_seen / len(train_set),
    "hours": train_seconds / 3600, "stop_reason": stop_reason,
    "best_step": best["step"], "best_val_loss": best["val_loss"], "best_val_ppl": best["val_ppl"],
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if n_gpus else None,
}
save_json(training_summary, RESULTS / "training_summary.json")

fig = plot_history(history, metrics=("loss",), x="step", title="Cross-entropy per token (no label smoothing)")
savefig(fig, FIGURES / "loss_curves.png")
fig = plot_history(history, metrics=("rouge1", "rouge2", "rougeLsum"), x="step",
                   title=f"Greedy summaries of {CFG['val_monitor_size']} validation articles during training")
for ax, name in zip(fig.axes, ("ROUGE-1", "ROUGE-2", "ROUGE-Lsum")):
    ax.set_title(name)
savefig(fig, FIGURES / "val_rouge.png")

from IPython.display import Image, display
display(Image(str(FIGURES / "loss_curves.png")), Image(str(FIGURES / "val_rouge.png")))
print(json.dumps(training_summary, indent=2))

## 6. Best checkpoint and decoding choice (validation only)

We restore the weights with the lowest validation loss, not the last ones. Then we compare plain **greedy**
decoding (required by the brief) with greedy + **trigram blocking** (never repeat a trigram) on validation
articles. The test set plays no part in this choice.

In [ ]:
stopper.restore(model)          # the best validation-loss checkpoint
model.eval()
print(f"restored the checkpoint from step {best['step']:,} (validation loss {best['val_loss']:.3f})")

choice_idx = val_order[: CFG["decode_choice_size"]]
choice_src = data["validation"]["src"][choice_idx]
choice_refs = [val_refs[i] for i in choice_idx]
DECODINGS = {"Greedy": False, "Greedy + trigram blocking": True}   # name -> block_trigrams

decoding_choice = {}
for name, block in DECODINGS.items():
    preds = q3_train.summarize_arrays(model, choice_src, tokenizer, SPECIAL, device, CFG["eval_batch_size"],
                                      CFG["max_summary_len"], block_trigrams=block, desc=name)
    decoding_choice[name] = q3_eval.rouge_means(q3_eval.rouge_per_example(preds, choice_refs))
save_json(decoding_choice, RESULTS / "decoding_choice_validation.json")

MAIN = max(decoding_choice, key=lambda name: decoding_choice[name]["rougeLsum"])
for name, scores in decoding_choice.items():
    print(f"{name:<27}" + "  ".join(f"{q3_eval.ROUGE_NAMES[k]} {v:.2f}" for k, v in scores.items()))
print(f"\nmain decoding (higher validation ROUGE-Lsum): {MAIN}")

## 7. ROUGE on the test set

Summaries are generated for **all 11,490 test articles** without teacher forcing, converted from ids to text,
written one sentence per line, and scored against the original highlights with `rouge_score`
(`use_stemmer=True`, F1). ROUGE-L (whole summary as one sequence) and ROUGE-Lsum (sentence-level, needs the
newlines) are reported separately. Lead-3 (first three article sentences) is computed with the same scorer.

In [ ]:
if not SMOKE:
    assert len(test_refs) == 11490, "real results must use the full test set"

test_preds, test_rouge_rows, test_rouge = {}, {}, {}
for name, block in DECODINGS.items():
    t0 = time.time()
    test_preds[name] = q3_train.summarize_arrays(model, data["test"]["src"], tokenizer, SPECIAL, device,
                                                 CFG["eval_batch_size"], CFG["max_summary_len"],
                                                 block_trigrams=block, desc=f"test, {name}")
    print(f"{name}: {len(test_preds[name]):,} summaries in {(time.time() - t0) / 60:.1f} min")

split_sentences, splitter_name = q3_eval.sentence_splitter()
test_preds["Lead-3"] = [q3_eval.lead3(article, split_sentences) for article in test_articles]

for name, preds in test_preds.items():
    test_rouge_rows[name] = q3_eval.rouge_per_example(preds, test_refs)
    test_rouge[name] = q3_eval.rouge_summary(test_rouge_rows[name])

save_jsonl(({"id": test_ids[i], **{name: test_preds[name][i] for name in DECODINGS},
             **{f"{name} {k}": round(test_rouge_rows[name][i][k], 2) for name in DECODINGS for k in q3_eval.ROUGE_TYPES}}
            for i in range(len(test_ids))), RESULTS / "test_predictions.jsonl")
rouge_table = {"Lead-3 (brief)": {k: {"mean": v} for k, v in q3_eval.LEAD3_BRIEF.items()},
               "Lead-3 (our run)": test_rouge["Lead-3"], **{name: test_rouge[name] for name in DECODINGS}}
save_json({"n_test": len(test_refs), "sentence_splitter": splitter_name, "main_decoding": MAIN,
           "scores": rouge_table}, RESULTS / "rouge.json")

print(f"\nTest set, {len(test_refs):,} pairs, F1 x 100 (95% bootstrap CI)")
for name, scores in rouge_table.items():
    print(f"{name:<27}" + "  ".join(
        f"{q3_eval.ROUGE_NAMES[k]} {s['mean']:.2f}" + (f" [{s['ci_low']:.2f}, {s['ci_high']:.2f}]" if "ci_low" in s else "")
        for k, s in scores.items()))

bars = {name: {q3_eval.ROUGE_NAMES[k]: test_rouge[name][k]["mean"] for k in q3_eval.ROUGE_TYPES}
        for name in ("Lead-3", *DECODINGS)}
plot_grouped_bars(bars, ylabel="F1 x 100", title="Test-set ROUGE", out_path=FIGURES / "rouge_comparison.png")
display(Image(str(FIGURES / "rouge_comparison.png")))

## 8. Qualitative analysis

Simple checks on every test summary: length, repeated word trigrams, numbers that do not occur in the article
(a rough sign of invented facts) and the share of bigrams not copied from the article (how abstractive the output
is). Then at least five test articles with reference and generated summaries, chosen to include failures.

In [ ]:
text_checks = {}
check_rows = {}
for name, texts in [("Reference", test_refs), ("Lead-3", test_preds["Lead-3"]),
                    *[(n, test_preds[n]) for n in DECODINGS]]:
    check_rows[name] = [q3_eval.text_checks(t, a) for t, a in zip(texts, test_articles)]
    text_checks[name] = q3_eval.aggregate_checks(check_rows[name])
save_json(text_checks, RESULTS / "text_checks.json")
for name, agg in text_checks.items():
    print(f"{name:<27}" + "  ".join(f"{k} {v:.1f}" for k, v in agg.items()))

# Pick examples from plain greedy output (where failures are easiest to see) and show both decodings
selection_rows = [{**test_rouge_rows["Greedy"][i], **check_rows["Greedy"][i]} for i in range(len(test_refs))]
median_lsum = float(np.median([row["rougeLsum"] for row in test_rouge_rows["Greedy"]]))
qualitative = []
for reason, i in q3_eval.pick_qualitative(selection_rows, seed=CFG["seed"]):
    greedy_text, blocked_text = test_preds["Greedy"][i], test_preds["Greedy + trigram blocking"][i]
    qualitative.append({
        "reason": reason, "index": i, "id": test_ids[i],
        "article_start": " ".join(test_articles[i].split()[:120]) + " ...",
        "reference": test_refs[i], "greedy": greedy_text, "blocked": blocked_text,
        "scores": {name: test_rouge_rows[name][i] for name in DECODINGS},
        "notes": q3_eval.describe_example(greedy_text, blocked_text, test_refs[i], test_articles[i],
                                          test_rouge_rows["Greedy"][i]["rougeLsum"], median_lsum),
    })
save_json(qualitative, RESULTS / "qualitative_examples.json")

for k, ex in enumerate(qualitative, 1):
    print(f"\n=== Example {k}: {ex['reason']} (id {ex['id']}) ===")
    print("ARTICLE:", ex["article_start"][:500])
    print("REFERENCE:\n" + ex["reference"])
    print("GREEDY:\n" + (ex["greedy"] or "(empty)"))
    print("GREEDY + TRIGRAM BLOCKING:\n" + (ex["blocked"] or "(empty)"))
    print("NOTES:", " ".join(ex["notes"]))

## 9. Encoder-decoder attention

Cross-attention of the **last decoder layer**, recorded at each step while the model generates the summary
greedily (no teacher forcing), **averaged over heads** (a per-head version follows). Rows are generated tokens,
columns are article tokens; we show the 60-token window of the article that gets the most attention.

In [ ]:
import matplotlib.pyplot as plt

banned_ids = [SPECIAL.pad, SPECIAL.bos, SPECIAL.unk]


def generate_with_attention(src_rows):
    # Greedy summaries of a few articles plus the last decoder layer's cross-attention, per article:
    # (generated ids up to [SEP], weights [heads, generated, article tokens], article ids)
    src = q3_data.trim_padding(torch.from_numpy(src_rows.astype(np.int64))).to(device)
    with autocast(device):
        new_tokens, attention = q3_model.greedy_summarize(
            model, src, SPECIAL.bos, SPECIAL.eos, SPECIAL.pad, max_len=CFG["max_summary_len"],
            banned_ids=banned_ids, record_attention=True)
    results = []
    for b in range(src.size(0)):
        ids = new_tokens[b].tolist()
        n = ids.index(SPECIAL.eos) + 1 if SPECIAL.eos in ids else len(ids)
        src_len = int((src[b] != SPECIAL.pad).sum())
        results.append((ids[:n], attention[b, :, :n, :src_len].cpu().numpy(), src[b, :src_len].tolist()))
    return results


WINDOW, MAX_ROWS = 60, 40
attention_examples, figure_paths = [], {}
# Example 1: the best summary (clearest alignment). Example 2: a failure with a repeated phrase, if there is one.
heatmap_examples = [qualitative[0], next((e for e in qualitative if e["reason"] == "repeated phrase"), qualitative[1])]
for k, ex in enumerate(heatmap_examples, 1):
    gen_ids, weights, src_ids = generate_with_attention(data["test"]["src"][ex["index"]:ex["index"] + 1])[0]
    gen_tokens = q3_data.tokens_for_plot(gen_ids, tokenizer)[:MAX_ROWS]
    src_tokens = q3_data.tokens_for_plot(src_ids, tokenizer)
    mean_weights = weights.mean(axis=0)[: len(gen_tokens)]                 # average over heads
    win_weights, win_tokens, win_start = attention_window(mean_weights, src_tokens, width=WINDOW)
    title = f"Example {k} ({ex['reason']}), last decoder layer, mean over {CFG['n_heads']} heads"
    path = FIGURES / f"cross_attention_example{k}.png"
    plot_attention(win_weights, win_tokens, gen_tokens, out_path=path, title=title)
    figure_paths[f"attention_example_{k}"] = path
    attention_examples.append({"title": title, "figure": f"attention_example_{k}", "start": win_start,
                               "width": len(win_tokens), "id": ex["id"]})
    np.save(RESULTS / f"cross_attention_example{k}.npy", weights)          # [heads, generated, article]
    if k == 1:   # every head separately, same window
        rows = 2 if CFG["n_heads"] > 4 else 1
        cols = math.ceil(CFG["n_heads"] / rows)
        fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows), squeeze=False)
        for h, ax in enumerate(axes.flat):
            if h < CFG["n_heads"]:
                head = weights[h][: len(gen_tokens), win_start:win_start + len(win_tokens)]
                plot_attention(head, win_tokens, gen_tokens, ax=ax, title=f"head {h + 1}")
            else:
                ax.axis("off")
        fig.suptitle("Example 1, last decoder layer, each head", fontweight="bold")
        fig.tight_layout()
        savefig(fig, FIGURES / "cross_attention_heads.png")
    print(f"example {k}: {ex['id']}, window starts at article token {win_start}")
    display(Image(str(path)))
display(Image(str(FIGURES / "cross_attention_heads.png")))

In [ ]:
# Where in the article does the decoder look? Attention mass by article position over many test summaries.
att_rows, att_gen_tokens, att_src_tokens, first100 = [], [], [], []
n_att = min(CFG["attention_articles"], len(test_refs))
for start in range(0, n_att, CFG["eval_batch_size"]):
    for gen_ids, weights, src_ids in generate_with_attention(data["test"]["src"][start:min(start + CFG["eval_batch_size"], n_att)]):
        att_rows.append(weights.mean(axis=0))
        att_gen_tokens.append(q3_data.tokens_for_plot(gen_ids, tokenizer))
        att_src_tokens.append(q3_data.tokens_for_plot(src_ids, tokenizer))
        first100.append(min(100, len(src_ids)) / len(src_ids))

position_bins = q3_eval.attention_by_position(att_rows, bin_size=50, max_len=CFG["src_len"])
bin_labels = [f"{s}-{min(s + 49, CFG['src_len'] - 1)}" for s in range(0, CFG["src_len"], 50)]
plot_grouped_bars({"cross-attention": dict(zip(bin_labels, position_bins))}, ylabel="% of attention mass",
                  xlabel="article token position", title=f"Last-layer cross-attention over {n_att} test summaries",
                  out_path=FIGURES / "attention_by_position.png")
attention_info = {
    "window": WINDOW, "examples": attention_examples, "n_articles": n_att, "position_bins": position_bins,
    "first100_share_of_source": 100 * float(np.mean(first100)),
    "copy_alignment": q3_eval.copy_alignment(att_rows, att_gen_tokens, att_src_tokens),
}
save_json(attention_info, RESULTS / "attention_stats.json")
print(f"attention on the first 100 article tokens: {position_bins[0] + position_bins[1]:.1f}% "
      f"(they are {attention_info['first100_share_of_source']:.1f}% of the article tokens)")
print(f"generated tokens whose most-attended article token is the same token: {attention_info['copy_alignment']:.1f}%")
display(Image(str(FIGURES / "attention_by_position.png")))

## 10. Report, deliverables and output ZIP

`REPORT.md` is written from the numbers above (settings, data sizes, losses, ROUGE, checks, examples and
figures), so it always matches this run. The ZIP holds `REPORT.md` and `outputs/` (results and figures),
without checkpoints or data.

In [ ]:
figure_paths.update({
    "loss_curves": FIGURES / "loss_curves.png", "val_rouge": FIGURES / "val_rouge.png",
    "rouge_comparison": FIGURES / "rouge_comparison.png", "attention_heads": FIGURES / "cross_attention_heads.png",
    "attention_by_position": FIGURES / "attention_by_position.png",
})
run_info = json.loads((RESULTS / "run_info.json").read_text())["environment"]
report_values = {
    "smoke": SMOKE, "cfg": CFG, "data_info": data_info, "params": params, "training": training_summary,
    "history": history.records, "sanity": checks,
    "peak_lr": q3_train.noam_lr(CFG["warmup_steps"], CFG["d_model"], CFG["warmup_steps"]),
    "precision": "fp16 autocast with loss scaling" if gpu_name != "CPU" and torch.cuda.get_device_capability()[0] < 8
                 else ("bf16 autocast" if gpu_name != "CPU" else "fp32 (CPU)"),
    "batch_size_total": batch_size_total, "hardware": f"{use_gpus} x {gpu_name}",
    "decoding_choice": decoding_choice, "decode_choice_size": len(choice_idx), "main_system": MAIN,
    "test_rouge": {"Lead-3 (our run)": test_rouge["Lead-3"], **{name: test_rouge[name] for name in DECODINGS}},
    "n_test": len(test_refs), "splitter": splitter_name, "text_checks": text_checks,
    "qualitative": qualitative, "attention": attention_info, "figures": figure_paths,
    "environment": f"Python {run_info['python']}, PyTorch {run_info['torch']}, {gpu_name}",
}
report_path = q3_report.write_report((OUT if SMOKE else Q3) / "REPORT.md", report_values)

deliverables = {
    "source code (data, model, training, evaluation)": ["Question_3.ipynb", "q3_data.py", "q3_model.py",
                                                        "q3_train.py", "q3_eval.py", "q3_report.py"],
    "README with run instructions and libraries": ["README.md", "requirements.txt"],
    "report": [str(report_path.relative_to(Q3))],
    "tokenizer, truncation, data sizes": [str((RESULTS / "data_info.json").relative_to(Q3))],
    "model configuration and parameter count": [str((RESULTS / "run_info.json").relative_to(Q3))],
    "checks for the pitfalls": [str((RESULTS / "sanity_checks.json").relative_to(Q3))],
    "training and validation loss curves": [str((FIGURES / "loss_curves.png").relative_to(Q3)),
                                            str((RESULTS / "history.json").relative_to(Q3))],
    "ROUGE-1/2/L/Lsum on the test set": [str((RESULTS / "rouge.json").relative_to(Q3)),
                                         str((RESULTS / "test_predictions.jsonl").relative_to(Q3))],
    ">= 5 qualitative examples incl. failures": [str((RESULTS / "qualitative_examples.json").relative_to(Q3))],
    "cross-attention heatmap": [str((FIGURES / "cross_attention_example1.png").relative_to(Q3))],
}
status = {item: all((Q3 / f).exists() for f in files) for item, files in deliverables.items()}
save_json({"files": deliverables, "present": status}, RESULTS / "deliverables.json")
for item, ok in status.items():
    print(f"{'ok     ' if ok else 'MISSING'}  {item}")

zip_path = (Path("/kaggle/working") if ON_KAGGLE else Q3) / ("q3_outputs_smoke.zip" if SMOKE else "q3_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(report_path, report_path.relative_to(Q3))
    for f in sorted(OUT.rglob("*")):
        if f.is_file() and "checkpoints" not in f.parts and f != report_path:
            zf.write(f, f.relative_to(Q3))
print(f"\nreport: {report_path}\nzip:    {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB, no checkpoints, no data)")

## 11. Try it on any article (optional)

In [ ]:
def summarize_article(text, block_trigrams=DECODINGS[MAIN]):
    ids = q3_data.truncate(tokenizer(q3_data.prepare_article(text))["input_ids"], CFG["src_len"])
    src = torch.tensor([ids], device=device)
    with autocast(device):
        new_tokens, _ = q3_model.greedy_summarize(model, src, SPECIAL.bos, SPECIAL.eos, SPECIAL.pad,
                                                  max_len=CFG["max_summary_len"], banned_ids=banned_ids,
                                                  block_trigrams=block_trigrams, sentence_id=SPECIAL.nl)
    return q3_data.ids_to_text(new_tokens[0].cpu(), tokenizer, SPECIAL)


print(summarize_article(test_articles[0]))